# 第 49 章 · Latent Diffusion 与 Rectified Flow：在压缩空间里扩散，走直线少走几步

配套讲义：[在线版](https://wikieden.github.io/beyond-transformer/lessons/49-latent-diffusion-flow.html) · 仓库里 `beyond-transformer/notes/lessons/49-latent-diffusion-flow.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。只用 NumPy，不需要 GPU。练习题不在这里算，先在讲义里自己做。

In [ ]:
import numpy as np

np.set_printoptions(precision=4, suppress=True)  # 打印保留 4 位小数，不用科学计数法
print("numpy", np.__version__)

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 公式 | 写法 | 跑出来 |
|---|---|---|---|
| 第一步 压缩多少 | $(H/f)(W/f)c$ | `latent_size(512, 8, 4)` | 64 × 64 × 4 = 16384，小 48 倍 |
| 第二步 交叉注意力 | softmax(QKᵀ/√d)V | `cross_attention(Q, K, V)` | A：[0.164, 0.673, 0.164] → [0.673, 0.164]；B：[0.164, 0.673] |
| 第三步 RF 训练 | 式 1 | `rf_example(-1, 0.8, 0.25, 1.5)` | $X_t$ = −0.55，目标 1.8，loss 0.09 |
| 第四步 Euler | $Z + v/N$ | `euler(z, v, dt)` | −0.55 + 0.75 × 1.8 = 0.8 |
| 第四步 完美速度场 | $\mathbb{E}[X_1 - X_0 \mid X_t]$ | `perfect_velocity(z, t, data)` | 两个数据点 ±1 时，z = 0、t = 0.5 处速度为 0（交叉点取平均） |
| 第四步 步数 | Euler N 步 | `generate(0.3, [-1, 1], N)` | 1 步落在 0（两个数据点正中间，哪个都不是）；2 步 0.291；10 步到 1.0 |

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

In [ ]:
K = np.array([[0, 0], [1, 0], [0, 1.0]])  # "一只"、"红色"、"猫"：两维分别是 颜色、动物（编的）
V = K.copy()
Q = np.array([[2, 0], [0, 2.0]])  # 位置 A 问颜色，位置 B 问是什么

**函数 `latent_size`**：side × side 的彩色图压成 (side/f) × (side/f) × c：返回边长、个数、比像素小几倍。

In [ ]:
def latent_size(side, f, c):
    """side × side 的彩色图压成 (side/f) × (side/f) × c：返回边长、个数、比像素小几倍。"""
    h = side // f
    return {"h": h, "numbers": h * h * c, "ratio": side * side * 3 / (h * h * c)}

**函数 `softmax`**：按行 softmax。

In [ ]:
def softmax(x):
    """按行 softmax。"""
    e = np.exp(x - x.max(axis=-1, keepdims=True))
    return e / e.sum(axis=-1, keepdims=True)

**函数 `cross_attention`**：交叉注意力：Q 来自图像位置，K、V 来自文字 token。

In [ ]:
def cross_attention(Q, K, V):
    """交叉注意力：Q 来自图像位置，K、V 来自文字 token。"""
    w = softmax(Q @ K.T / np.sqrt(K.shape[1]))
    return w, w @ V

**函数 `rf_example`**：Rectified Flow 的一次训练：直线上的位置、目标速度、平方误差。

In [ ]:
def rf_example(x0, x1, t, v_hat):
    """Rectified Flow 的一次训练：直线上的位置、目标速度、平方误差。"""
    xt = t * x1 + (1 - t) * x0
    target = x1 - x0
    return xt, target, (target - v_hat) ** 2

**函数 `euler`**：Euler 一步：按当前速度走 dt。

In [ ]:
def euler(z, v, dt):
    """Euler 一步：按当前速度走 dt。"""
    return z + dt * v

**函数 `perfect_velocity`**：完美速度场 E[X1 − X0 | X_t = z]：噪声是标准高斯，数据是给定的几个点。

In [ ]:
def perfect_velocity(z, t, data):
    """完美速度场 E[X1 − X0 | X_t = z]：噪声是标准高斯，数据是给定的几个点。"""
    data = np.atleast_1d(np.asarray(data, float))
    logw = -((z - t * data) ** 2) / (2 * (1 - t) ** 2)
    w = np.exp(logw - logw.max())
    w /= w.sum()
    return float((w * (data - z)).sum() / (1 - t))

**函数 `generate`**：从噪声 z0 出发，用完美速度场走 n_steps 步 Euler，到 t = 1。

In [ ]:
def generate(z0, data, n_steps):
    """从噪声 z0 出发，用完美速度场走 n_steps 步 Euler，到 t = 1。"""
    z = z0
    for i in range(n_steps):
        z = euler(z, perfect_velocity(z, i / n_steps, data), 1 / n_steps)
    return z

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第一步 压缩：512 × 512 × 3，f = 8，4 个通道")
print(latent_size(512, 8, 4))

In [ ]:
print("\n第二步 交叉注意力：文字 一只 / 红色 / 猫")
w, out = cross_attention(Q, K, V)
print("权重：\n", w)
print("输出：\n", out)

In [ ]:
print("\n第三步 RF 训练：X0 = −1（噪声），X1 = 0.8（图），t = 0.25，网络猜 1.5")
xt, target, loss = rf_example(-1, 0.8, 0.25, 1.5)
print(f"X_t = {xt:.2f}，目标速度 {target:.1f}，loss = {loss:.2f}")

In [ ]:
print("\n第四步 Euler：从 −0.55 按速度 1.8 走完剩下的 0.75")
print(round(euler(-0.55, 1.8, 0.75), 3))
print("两个数据点 ±1：z = 0、t = 0.5 处的完美速度 =", perfect_velocity(0.0, 0.5, [-1, 1]))
for n in (1, 2, 10):
    print(f"从 z0 = 0.3 出发走 {n:2d} 步 →", round(generate(0.3, [-1, 1], n), 3))

## 3. 代码里才看得到的

- RF 的约定是 t = 0 噪声、t = 1 图；DDPM 是 t = 0 图、t = T 噪声；SD3 论文又用了"t = 0 是图"的写法。读代码时先看清楚哪头是噪声。
- 完美速度场里有 $1/(1-t)$，t = 1 时除以 0；Euler 最后一步只在 t = 1 − 1/N 取速度，不会碰到 t = 1。

## 4. 自己改一改

- 把 `cross_attention` 里位置 A 的 Q 放大成 [4, 0]，看它对"红色"的权重是不是更接近 1。
- 用 `perfect_velocity` 和 `euler` 从 z = −1 出发，分别走 1 步、10 步，看终点差多少。

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 先画小稿再放大：压成潜空间，扩散在里面；f 取八上下，太小慢、太大糊。
> 画师边画看订单：图像当 Q，文字当 K 和 V。
> 噪声连图走直线，网络只猜速度：终点减起点。
> 欧拉按速走几步，直了一步到；线交改道变弯，reflow 再拉直。